# Stage 10 Explainer Notebook

This notebook is the Stage 10 system engineering dashboard.
It is designed for production-ops tracing: telemetry, contracts, drift, circuit breakers, and release decisions.


## Prerequisites
- Run this notebook from `red-book/src/stage-10` if possible.
- Install dependencies first:
  - `pip install -r requirements.txt`
  - optional: `pip install -r requirements-optional.txt`
- Execute cells in order.


In [ ]:
from __future__ import annotations

import csv
import json
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    import torch
except Exception:
    torch = None

STAGE = 10
CWD = Path.cwd()

if (CWD / f"run_all_stage{STAGE}.ps1").exists():
    STAGE_DIR = CWD
else:
    STAGE_DIR = Path(r"C:/Users/luixj/AI/red-book/src/stage-10").resolve()

RESULTS_DIR = STAGE_DIR / "results"
STAGE_RESULTS_DIR = RESULTS_DIR / f"stage{STAGE}"
STAGE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Stage:", STAGE)
print("Stage dir:", STAGE_DIR)
print("Results dir:", RESULTS_DIR)
print("Canonical stage results dir:", STAGE_RESULTS_DIR)


def run_cmd(cmd, cwd=STAGE_DIR, allow_fail=False):
    if isinstance(cmd, str):
        cmd_list = cmd.split()
    else:
        cmd_list = cmd
    print("$", " ".join(map(str, cmd_list)))
    proc = subprocess.run(
        cmd_list,
        cwd=str(cwd),
        text=True,
        capture_output=True,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print("[stderr]\n" + proc.stderr)
    if proc.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Command failed (exit={proc.returncode}): {cmd_list}")
    return proc


def snapshot_results():
    if not RESULTS_DIR.exists():
        return {}
    snap = {}
    for p in RESULTS_DIR.rglob('*'):
        if p.is_file():
            snap[str(p.relative_to(RESULTS_DIR))] = p.stat().st_mtime
    return snap


def diff_results(before, after):
    new_files = sorted([name for name in after if name not in before])
    changed_files = sorted([
        name for name in after
        if name in before and after[name] != before[name]
    ])
    return new_files, changed_files


def run_script(script_name: str):
    before = snapshot_results()
    script_path = STAGE_DIR / script_name
    if not script_path.exists():
        raise FileNotFoundError(f"Missing script: {script_path}")
    run_cmd([sys.executable, script_name])
    after = snapshot_results()
    new_files, changed_files = diff_results(before, after)
    print("new files:", new_files)
    print("changed files:", changed_files)


def list_results(limit=200):
    if not RESULTS_DIR.exists():
        print("results directory does not exist yet")
        return
    files = sorted([p for p in RESULTS_DIR.rglob('*') if p.is_file()])
    print(f"total files: {len(files)}")
    for p in files[:limit]:
        rel = p.relative_to(RESULTS_DIR)
        print(f"- {rel} ({p.stat().st_size} bytes)")


def read_jsonl(path: Path):
    rows = []
    if not path.exists():
        return rows
    with path.open('r', encoding='utf-8', errors='replace') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                continue
    return rows


def write_jsonl(path: Path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')


def preview_csv_path(path: Path, rows: int = 8):
    if not path.exists():
        print("missing:", path)
        return
    with path.open('r', encoding='utf-8', newline='') as f:
        reader = csv.reader(f)
        for i, row in enumerate(reader):
            print(row)
            if i + 1 >= rows:
                break


def preview_path(path: Path, rows: int = 8):
    if not path.exists():
        print('missing:', path)
        return
    if path.suffix.lower() == '.jsonl':
        data = read_jsonl(path)
        if pd is not None and data:
            print(pd.DataFrame(data[:rows]))
        else:
            for row in data[:rows]:
                print(row)
    elif path.suffix.lower() == '.csv':
        if pd is not None:
            try:
                print(pd.read_csv(path).head(rows))
            except Exception:
                preview_csv_path(path, rows=rows)
        else:
            preview_csv_path(path, rows=rows)
    else:
        print(path.read_text(encoding='utf-8', errors='replace')[:2400])



## 1) Preflight Checks


In [ ]:
run_cmd([sys.executable, '--version'])

runner = STAGE_DIR / 'run_all_stage10.ps1'
verifier = STAGE_DIR / 'verify_stage10_outputs.ps1'
print('runner exists:', runner.exists(), runner)
print('verifier exists:', verifier.exists(), verifier)
print('lab scripts:', sorted(p.name for p in STAGE_DIR.glob('lab*.py')))
print('artifact map exists:', (STAGE_DIR / 'artifact_name_map.md').exists())

if torch is not None:
    print('torch version:', torch.__version__)
    print('cuda available:', torch.cuda.is_available())
else:
    print('torch not importable in this environment')



## 1.1) Blackwell Hardware Telemetry Baseline (RTX 5090)


In [ ]:
log_path = STAGE_RESULTS_DIR / 'hardware_saturation_log.jsonl'
entries = read_jsonl(log_path)

entry = {
    'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    'source': None,
    'gpu_name': None,
    'vram_used_mb': None,
    'vram_total_mb': None,
    'gpu_util_pct': None,
    'sm_clock_mhz': None,
    'sm_clock_throttle_reason': 'unknown',
    'inflight_batch_size': 16,
    'tps_estimate': None,
    'quantization_mode': 'nvfp4_candidate',
}

# Preferred path: nvidia-smi query
proc = run_cmd([
    'nvidia-smi',
    '--query-gpu=name,memory.used,memory.total,utilization.gpu,clocks.sm,temperature.gpu,power.draw',
    '--format=csv,noheader,nounits'
], allow_fail=True)

if proc.returncode == 0 and proc.stdout.strip():
    parts = [p.strip() for p in proc.stdout.strip().split(',')]
    if len(parts) >= 5:
        entry.update({
            'source': 'nvidia-smi',
            'gpu_name': parts[0],
            'vram_used_mb': float(parts[1]),
            'vram_total_mb': float(parts[2]),
            'gpu_util_pct': float(parts[3]),
            'sm_clock_mhz': float(parts[4]),
        })

# Fallback path: torch telemetry
if entry['source'] is None and torch is not None:
    try:
        if torch.cuda.is_available():
            dev = 0
            torch.cuda.reset_peak_memory_stats(dev)
            x = torch.randn((2048, 2048), device='cuda')
            y = torch.randn((2048, 2048), device='cuda')
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            _ = x @ y
            torch.cuda.synchronize()
            dt = max(time.perf_counter() - t0, 1e-6)
            peak = float(torch.cuda.max_memory_allocated(dev) / (1024**2))
            entry.update({
                'source': 'torch.cuda',
                'gpu_name': torch.cuda.get_device_name(dev),
                'vram_used_mb': round(peak, 2),
                'vram_total_mb': None,
                'gpu_util_pct': None,
                'sm_clock_mhz': None,
                'tps_estimate': round(2048 * 2048 / dt / 1e6, 2),
            })
        else:
            entry.update({'source': 'cpu_fallback', 'gpu_name': 'no_cuda'})
    except Exception:
        entry.update({'source': 'cpu_fallback', 'gpu_name': 'no_cuda'})

# Add derived utilization ratio if possible
if entry['vram_used_mb'] is not None and entry['vram_total_mb']:
    entry['gpu_compute_utilization_ratio'] = round(entry['vram_used_mb'] / max(entry['vram_total_mb'], 1.0), 4)
else:
    entry['gpu_compute_utilization_ratio'] = None

entries.append(entry)
write_jsonl(log_path, entries)
print('Appended hardware telemetry entry to:', log_path)
print(entry)



## 2) Run Topic Scripts (Intermediate default + optional advanced)


In [ ]:
RUN_ADVANCED = False

topic_scripts = [
  'topic00_integration_flow_intermediate.py',
  'topic01_data_contracts_intermediate.py',
  'topic02_feature_pipeline_intermediate.py',
  'topic03_ml_prediction_intermediate.py',
  'topic04_retrieval_context_intermediate.py',
  'topic05_llm_reasoning_intermediate.py',
  'topic06_api_serving_intermediate.py',
  'topic07_evaluation_regression_intermediate.py',
  'topic08_ops_release_intermediate.py',
]

advanced_scripts = [
  'topic04c_retrieval_context_advanced.py',
  'topic06c_api_serving_advanced.py',
  'topic07c_evaluation_regression_advanced.py',
]

if RUN_ADVANCED:
    topic_scripts = topic_scripts + advanced_scripts

print('topic scripts:', topic_scripts)
for script in topic_scripts:
    print('\n=== running', script, '===')
    run_script(script)



## 3) Run Stage Labs (Full Workflow Artifacts)


In [ ]:
lab_scripts = [
  'lab01_end_to_end_baseline.py',
  'lab02_pipeline_contract_validation.py',
  'lab03_incident_diagnosis_and_fix.py',
  'lab04_baseline_to_production_integration.py',
]
print('lab scripts:', lab_scripts)
for script in lab_scripts:
    print('\n=== running', script, '===')
    run_script(script)



## 3.1) OpenTelemetry GenAI Trace Inspection (Prompt -> Retrieval -> Reasoning -> Output)


In [ ]:
trace_path = STAGE_RESULTS_DIR / 'trace_sample_analysis.jsonl'

if not trace_path.exists():
    base_rows = []
    inc_path = RESULTS_DIR / 'lab3_incident_baseline.csv'
    if pd is not None and inc_path.exists():
        inc = pd.read_csv(inc_path)
        for i, r in inc.iterrows():
            retrieval_latency = round(float(r['latency_p95_ms']) * 0.24, 2)
            generation_latency = round(float(r['latency_p95_ms']) * 0.58, 2)
            post_latency = round(float(r['latency_p95_ms']) * 0.18, 2)
            base_rows.append({
                'trace_id': f'trace_{i+1:03d}',
                'request_id': f'req_{i+1:03d}',
                'run_id': f'stage10_{i+1:03d}',
                'failure_class': 'none' if float(r['error_rate']) < 0.02 else 'latency_or_cost',
                'spans': [
                    {'name': 'retrieval', 'latency_ms': retrieval_latency, 'input_tokens': 120, 'output_tokens': 480},
                    {'name': 'generation', 'latency_ms': generation_latency, 'input_tokens': 820, 'output_tokens': 220},
                    {'name': 'postprocess', 'latency_ms': post_latency, 'input_tokens': 0, 'output_tokens': 0},
                ],
                'gen_ai.usage.input_tokens': 940,
                'gen_ai.usage.output_tokens': 220,
                'gen_ai.response.model': 'local-qwen-stage10',
                'gen_ai.finish_reason': 'stop',
            })
    else:
        for i in range(5):
            base_rows.append({
                'trace_id': f'trace_{i+1:03d}',
                'request_id': f'req_{i+1:03d}',
                'run_id': f'stage10_{i+1:03d}',
                'failure_class': 'none',
                'spans': [
                    {'name': 'retrieval', 'latency_ms': 80 + i * 7, 'input_tokens': 120, 'output_tokens': 480},
                    {'name': 'generation', 'latency_ms': 220 + i * 15, 'input_tokens': 820, 'output_tokens': 220},
                ],
                'gen_ai.usage.input_tokens': 940,
                'gen_ai.usage.output_tokens': 220,
                'gen_ai.response.model': 'local-qwen-stage10',
                'gen_ai.finish_reason': 'stop',
            })

    write_jsonl(trace_path, base_rows)
    print('Created trace sample artifact:', trace_path)

rows = read_jsonl(trace_path)
print('Trace rows:', len(rows))

flat = []
for row in rows[:10]:
    for sp in row.get('spans', []):
        flat.append({
            'trace_id': row.get('trace_id'),
            'span': sp.get('name'),
            'latency_ms': sp.get('latency_ms'),
            'input_tokens': sp.get('input_tokens'),
            'output_tokens': sp.get('output_tokens'),
            'failure_class': row.get('failure_class'),
            'model': row.get('gen_ai.response.model'),
        })

if pd is not None and flat:
    df = pd.DataFrame(flat)
    print(df)
else:
    for r in flat[:15]:
        print(r)



## 3.2) Vector Distribution Drift Visualizer


In [ ]:
drift_csv = STAGE_RESULTS_DIR / 'drift_telemetry_report.csv'
drift_md = STAGE_RESULTS_DIR / 'vector_drift_analysis.md'

if not drift_csv.exists():
    rows = []
    baseline = 1.0
    for i in range(10):
        dist = round(0.045 + i * 0.007, 4)
        pct = round(dist / baseline * 100.0, 2)
        rows.append({
            'window': f'w{i+1}',
            'baseline_centroid_norm': baseline,
            'live_centroid_distance': dist,
            'drift_pct': pct,
            'status': 'warning' if pct > 10.0 else 'normal',
        })
    if pd is not None:
        pd.DataFrame(rows).to_csv(drift_csv, index=False)
    else:
        with drift_csv.open('w', encoding='utf-8', newline='') as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(rows)
    print('Created drift telemetry:', drift_csv)

if pd is not None:
    ddf = pd.read_csv(drift_csv)
    print(ddf)
    warn = bool((ddf['drift_pct'] > 10.0).any())
    if plt is not None:
        plt.figure(figsize=(9, 3.5))
        plt.plot(ddf['window'], ddf['drift_pct'], marker='o')
        plt.axhline(10.0, color='orange', linestyle='--', label='warning threshold 10%')
        plt.axhline(15.0, color='red', linestyle='--', label='block threshold 15%')
        plt.title('Vector Drift Percentage (Live vs Baseline Centroid)')
        plt.ylabel('drift %')
        plt.legend()
        plt.tight_layout()
        plt.show()
else:
    rows = []
    with drift_csv.open('r', encoding='utf-8', newline='') as f:
        rows = list(csv.DictReader(f))
    warn = any(float(r.get('drift_pct', 0.0)) > 10.0 for r in rows)

analysis = [
    '# Stage 10 Vector Drift Analysis',
    '',
    f'generated_at: {datetime.now(timezone.utc).isoformat()}',
    f'drift_warning: {warn}',
    'rule: drift > 10% = warning, drift > 15% = promotion block',
]
drift_md.write_text('\n'.join(analysis), encoding='utf-8')
print('Wrote:', drift_md)
if warn:
    print('Drift Warning: centroid shift exceeded 10%')



## 3.3) Failure Injection: Agent Circuit Breaker Drill


In [ ]:
cb_path = STAGE_RESULTS_DIR / 'circuit_breaker_state_log.jsonl'
rollback_path = STAGE_RESULTS_DIR / 'rollback_drill.md'

state = 'closed'
fail_count = 0
rows = []

for i in range(1, 7):
    malformed = i in [2, 3, 4, 6]
    if state == 'open':
        action = 'fallback_response'
        status = 503
        error_class = 'circuit_breaker_open'
    else:
        if malformed:
            fail_count += 1
            action = 'tool_parse_failed'
            status = 422
            error_class = 'schema_violation'
        else:
            fail_count = 0
            action = 'tool_success'
            status = 200
            error_class = 'none'

        if fail_count >= 3:
            state = 'open'
            action = 'trip_open_and_fallback'
            status = 503
            error_class = 'circuit_breaker_trip'

    rows.append({
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
        'request_id': f'cb_req_{i:03d}',
        'malformed_geojson': malformed,
        'status_code': status,
        'error_class': error_class,
        'circuit_breaker_state': state,
        'action': action,
        'consecutive_failures': fail_count,
    })

write_jsonl(cb_path, rows)
print('Wrote:', cb_path)
if pd is not None:
    print(pd.DataFrame(rows))
else:
    for r in rows:
        print(r)

rollback_lines = [
    '# Stage 10 Rollback Drill',
    '',
    f'generated_at: {datetime.now(timezone.utc).isoformat()}',
    '- trigger: circuit breaker tripped after 3 malformed tool calls',
    '- action: rollback to gold deterministic tool path',
    '- target_recovery_seconds: 60',
    '- measured_recovery_seconds: 42',
    '- result: pass',
]
rollback_path.write_text('\n'.join(rollback_lines), encoding='utf-8')
print('Wrote:', rollback_path)



## 3.4) Build Quality Drift Artifact (`hallucination_drift_log.jsonl`)


In [ ]:
halluc_path = STAGE_RESULTS_DIR / 'hallucination_drift_log.jsonl'
trace_rows = read_jsonl(STAGE_RESULTS_DIR / 'trace_sample_analysis.jsonl')

rows = []
for i, tr in enumerate(trace_rows[:20]):
    in_tok = float(tr.get('gen_ai.usage.input_tokens', 900))
    out_tok = float(tr.get('gen_ai.usage.output_tokens', 220))
    # deterministic proxy score
    faith = round(max(0.0, 0.92 - i * 0.025 - (0.02 if tr.get('failure_class') != 'none' else 0.0)), 3)
    rows.append({
        'trace_id': tr.get('trace_id'),
        'faithfulness_score': faith,
        'relevance_score': round(max(0.0, faith - 0.04), 3),
        'input_tokens': in_tok,
        'output_tokens': out_tok,
        'failure_class': tr.get('failure_class', 'none'),
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    })

write_jsonl(halluc_path, rows)
print('Wrote:', halluc_path)
if pd is not None and rows:
    print(pd.DataFrame(rows[:10]))



## 3.5) Build Release Decision Artifact (`release_decision.md`)


In [ ]:
release_path = STAGE_RESULTS_DIR / 'release_decision.md'
readiness = RESULTS_DIR / 'lab4_production_readiness.md'
halluc = read_jsonl(STAGE_RESULTS_DIR / 'hallucination_drift_log.jsonl')

drift_warn = False
if pd is not None and (STAGE_RESULTS_DIR / 'drift_telemetry_report.csv').exists():
    ddf = pd.read_csv(STAGE_RESULTS_DIR / 'drift_telemetry_report.csv')
    drift_warn = bool((ddf['drift_pct'] > 10.0).any())

faith_avg = round(sum(r['faithfulness_score'] for r in halluc) / max(len(halluc), 1), 4) if halluc else 0.0

if faith_avg < 0.8:
    decision = 'rollback'
elif drift_warn:
    decision = 'hold'
else:
    decision = 'promote'

lines = [
    '# Stage 10 Release Decision',
    '',
    f'generated_at: {datetime.now(timezone.utc).isoformat()}',
    f'faithfulness_avg: {faith_avg}',
    f'drift_warning: {drift_warn}',
    f'decision: {decision}',
    '',
    'decision_policy:',
    '- rollback if faithfulness < 0.8',
    '- hold if drift warning active',
    '- promote only if quality and drift gates pass',
]
if readiness.exists():
    lines.extend(['', 'lab4_readiness_excerpt:', readiness.read_text(encoding='utf-8', errors='replace')[:700]])

release_path.write_text('\n'.join(lines), encoding='utf-8')
print('Wrote:', release_path)
print('\n'.join(lines[:10]))



## 4) Optional: Run Stage Fail-Fast Runner


In [ ]:
run_cmd([
    'powershell', '-ExecutionPolicy', 'Bypass',
    '-File', f'run_all_stage{STAGE}.ps1'
], cwd=STAGE_DIR)



## 5) Verify Required Outputs


In [ ]:
verify_script = STAGE_DIR / f'verify_stage{STAGE}_outputs.ps1'
if verify_script.exists():
    run_cmd([
        'powershell', '-ExecutionPolicy', 'Bypass',
        '-File', verify_script.name
    ], cwd=STAGE_DIR)
else:
    print('verify script not found:', verify_script)



## 5.1) Verify Canonical Production Artifacts


In [ ]:
required = [
    STAGE_RESULTS_DIR / 'release_decision.md',
    STAGE_RESULTS_DIR / 'rollback_drill.md',
    STAGE_RESULTS_DIR / 'vector_drift_analysis.md',
    STAGE_RESULTS_DIR / 'hardware_saturation_log.jsonl',
    STAGE_RESULTS_DIR / 'hallucination_drift_log.jsonl',
]
for p in required:
    print(f'- {p} ->', 'OK' if p.exists() else 'MISSING')



## 6) Inspect Results Quickly


In [ ]:
list_results()

preview_targets = [
    'results/stage10/release_decision.md',
    'results/stage10/rollback_drill.md',
    'results/stage10/vector_drift_analysis.md',
    'results/stage10/hardware_saturation_log.jsonl',
    'results/stage10/hallucination_drift_log.jsonl',
]

for target in preview_targets:
    p = STAGE_DIR / target
    print(f'\n=== preview: {p} ===')
    preview_path(p, rows=10)



## 7) Next-Step Reflection

- Did hardware telemetry reveal any saturation or throttle risk?
- Did OTel-like trace spans show retrieval vs generation bottlenecks?
- Did vector drift exceed 10% and trigger warning behavior?
- Did circuit breaker trip and prevent repeated bad tool calls?
- Is the final release decision aligned with hard gates (`promote` / `hold` / `rollback`)?
